In [1]:
pip install yfinance matplotlib


Note: you may need to restart the kernel to use updated packages.


In [10]:
import yfinance as yf
import pandas as pd

df=yf.download("AAPL", start="2020-01-01")
df['SMA50']=df['Close'].rolling(window=50).mean()
df['SMA200']=df['Close'].rolling(window=200).mean()
df['Signal']=0.0
df.loc[df['SMA50']>df['SMA200'], 'Signal']=1.0

df['Market_Return']=df['Close'].pct_change()
df['Strategy_Return']=df['Signal'].shift(1)*df['Market_Return']
print(df)

print(f"Cummulative Market Return: {(1+df['Market_Return']).prod()-1}")
print(f"Cumulative Strategy Return: {(1 + df['Strategy_Return']).prod()-1}")

C:\Users\Manav Soni\AppData\Local\Temp\ipykernel_25800\1337402914.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df=yf.download("AAPL", start="2020-01-01")
[*********************100%***********************]  1 of 1 completed

Price            Close        High         Low        Open     Volume  \
Ticker            AAPL        AAPL        AAPL        AAPL       AAPL   
Date                                                                    
2020-01-02   72.468254   72.528574   71.223252   71.476592  135480400   
2020-01-03   71.763718   72.523746   71.539330   71.696160  146322800   
2020-01-06   72.335564   72.374169   70.634547   70.885479  118387200   
2020-01-07   71.995361   72.600968   71.775796   72.345212  108872000   
2020-01-08   73.153496   73.455095   71.698581   71.698581  132079200   
...                ...         ...         ...         ...        ...   
2025-12-18  272.190002  273.630005  266.950012  273.609985   51630700   
2025-12-19  273.670013  274.600006  269.899994  272.149994  144632000   
2025-12-22  270.970001  273.880005  270.510010  272.859985   36571800   
2025-12-23  272.359985  272.500000  269.559998  270.839996   29642000   
2025-12-24  273.809998  275.429993  272.200012  272

In [46]:
import yfinance as yf
import pandas as pd
import numpy as np

df=yf.download("AAPL", start="2020-01-01")

df['SMA50']=df['Close'].rolling(window=2).mean()
df['SMA200']=df['Close'].rolling(window=5).mean()
df['Daily_Return']=df['Close'].pct_change()


rolling_mean=df['Daily_Return'].rolling(window=50).mean()
rolling_std=df['Daily_Return'].rolling(window=50).std()
df['Rolling_Sharpe']=np.sqrt(252)*(rolling_mean/rolling_std)

df['Signal']=0.0
df.loc[((df['SMA50']>df['SMA200']) & (df['Rolling_Sharpe']>0)), 'Signal']=1.0
df.loc[((df['SMA50']<df['SMA200']) & (df['Rolling_Sharpe']<0)), 'Signal']=-1.0

df['Market_Return']=df['Close'].pct_change()
df['Strategy_Return']=df['Signal'].shift(1)*df['Market_Return']
print(df)

df_clean=df.dropna().copy()
mean_return=df_clean['Strategy_Return'].mean()
std_dev=df_clean['Strategy_Return'].std()
strategy_sharpe=np.sqrt(252)*(mean_return/std_dev)



print(f"Overall Sharpe Ratio: {strategy_sharpe}")
print(f"Cummulative Market Return: {100*((1+df['Market_Return']).prod()-1)}")
print(f"Cumulative Strategy Return: {100*((1 + df['Strategy_Return']).prod()-1)}")

C:\Users\Manav Soni\AppData\Local\Temp\ipykernel_25800\841112315.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df=yf.download("AAPL", start="2020-01-01")
[*********************100%***********************]  1 of 1 completed

Price            Close        High         Low        Open     Volume  \
Ticker            AAPL        AAPL        AAPL        AAPL       AAPL   
Date                                                                    
2020-01-02   72.468246   72.528566   71.223244   71.476585  135480400   
2020-01-03   71.763741   72.523769   71.539352   71.696183  146322800   
2020-01-06   72.335556   72.374162   70.634539   70.885472  118387200   
2020-01-07   71.995361   72.600968   71.775796   72.345212  108872000   
2020-01-08   73.153496   73.455095   71.698581   71.698581  132079200   
...                ...         ...         ...         ...        ...   
2025-12-18  272.190002  273.630005  266.950012  273.609985   51630700   
2025-12-19  273.670013  274.600006  269.899994  272.149994  144632000   
2025-12-22  270.970001  273.880005  270.510010  272.859985   36571800   
2025-12-23  272.359985  272.500000  269.559998  270.839996   29642000   
2025-12-24  273.809998  275.429993  272.200012  272

In [20]:
import yfinance as yf
import pandas as pd
import numpy as np
df_a=yf.download("AAPL", start="2020-01-01")
df_m=yf.download("MSFT", start="2020-01-01")
df = pd.DataFrame(index=df_a.index)


df_a['Daily_Return']=df_a['Close'].pct_change()
df_m['Daily_Return']=df_m['Close'].pct_change()

df['AAPL_Daily_Return']=df_a['Daily_Return'].squeeze()
df['MSFT_Daily_Return']=df_m['Daily_Return'].squeeze()

df['AAPL_Close'] = df_a['Close'].squeeze()
df['MSFT_Close'] = df_m['Close'].squeeze()
df["Price_Ratio"]=df["AAPL_Close"]/df["MSFT_Close"]
df["Mean"]=df["Price_Ratio"].rolling(window=20).mean()
df["Std"]=df["Price_Ratio"].rolling(window=20).std()

df["Z_Score"]=(df["Price_Ratio"]-df["Mean"])/(df["Std"])
df["Sign_AAPL"]=0
df["Sign_MSFT"]=0

df.loc[df["Z_Score"]>2,"Sign_AAPL"]=-1
df.loc[df["Z_Score"]>2,"Sign_MSFT"]=1


df.loc[df["Z_Score"]<-2,"Sign_AAPL"]=1
df.loc[df["Z_Score"]<-2,"Sign_MSFT"]=-1

df['Strat_Ret'] = (df['Sign_AAPL'].shift(1) * df['AAPL_Daily_Return'] + 
                   df['Sign_MSFT'].shift(1) * df['MSFT_Daily_Return'])/ 2

print(df)
# 1. Drop NaNs to ensure clean calculations 
# (The first 21 days will be NaN due to the 20-day window and the 1-day shift)
clean_returns = df['Strat_Ret'].dropna()

# 2. Calculate the Mean and Standard Deviation of daily strategy returns
mean_return = clean_returns.mean()
std_dev = clean_returns.std()

# 3. Annualize the Sharpe Ratio
# We multiply by the square root of 252 (the number of trading days in a year)
if std_dev != 0:
    overall_sharpe = np.sqrt(252) * (mean_return / std_dev)
else:
    overall_sharpe = 0

# 4. Print results
print(f"\nOverall Strategy Sharpe Ratio: {overall_sharpe}")
print(f"Cumulative Strategy Return: {((1 + clean_returns).prod() - 1) * 100}")

# 1. Calculate the Cumulative Returns (Equity Curve)
# We start at 1 and multiply by (1 + daily_return)
df['Equity_Curve'] = (1 + clean_returns).cumprod()

# 2. Calculate the Running Maximum (the High-Water Mark)
df['High_Water_Mark'] = df['Equity_Curve'].cummax()

# 3. Calculate Drawdown
df['Drawdown'] = (df['Equity_Curve'] - df['High_Water_Mark']) / df['High_Water_Mark']

# 4. Find the Maximum Drawdown
max_drawdown = df['Drawdown'].min()

print(f"Maximum Drawdown: {max_drawdown * 100:.2f}%")

C:\Users\Manav Soni\AppData\Local\Temp\ipykernel_20640\4265608565.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_a=yf.download("AAPL", start="2020-01-01")
[*********************100%***********************]  1 of 1 completed
C:\Users\Manav Soni\AppData\Local\Temp\ipykernel_20640\4265608565.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_m=yf.download("MSFT", start="2020-01-01")
[*********************100%***********************]  1 of 1 completed

            AAPL_Daily_Return  MSFT_Daily_Return  AAPL_Close  MSFT_Close  \
Date                                                                       
2020-01-02                NaN                NaN   72.468254  152.505676   
2020-01-03          -0.009722          -0.012452   71.763733  150.606735   
2020-01-06           0.007968           0.002585   72.335556  150.996017   
2020-01-07          -0.004703          -0.009118   71.995369  149.619263   
2020-01-08           0.016086           0.015928   73.153481  152.002441   
...                       ...                ...         ...         ...   
2025-12-18           0.001288           0.016508  272.190002  483.980011   
2025-12-19           0.005437           0.004008  273.670013  485.920013   
2025-12-22          -0.009866          -0.002058  270.970001  484.920013   
2025-12-23           0.005130           0.003980  272.359985  486.850006   
2025-12-24           0.005324           0.002403  273.809998  488.019989   

           